## Introduction

TBD

 ## Python Imports

In [371]:
# Standard Library
import os
import sys
import warnings

from pathlib import Path

# Data Handling
import pandas as pd

# Suppress warnings
warnings.filterwarnings("ignore")

In [372]:
# Add the source subdirectory to the system path to allow import config from settings.py
current_directory = Path(os.getcwd())
website_base_directory = current_directory.parent.parent.parent
src_directory = website_base_directory / "src"
sys.path.append(str(src_directory)) if str(src_directory) not in sys.path else None

# Import settings.py
from settings import config

# Add configured directories from config to path
SOURCE_DIR = config("SOURCE_DIR")
sys.path.append(str(Path(SOURCE_DIR))) if str(Path(SOURCE_DIR)) not in sys.path else None

# Add other configured directories
BASE_DIR = config("BASE_DIR")
CONTENT_DIR = config("CONTENT_DIR")
POSTS_DIR = config("POSTS_DIR")
PAGES_DIR = config("PAGES_DIR")
PUBLIC_DIR = config("PUBLIC_DIR")
SOURCE_DIR = config("SOURCE_DIR")
DATA_DIR = config("DATA_DIR")
DATA_MANUAL_DIR = config("DATA_MANUAL_DIR")

## Python Functions

Here are the functions needed for this project:

* [load_data](/posts/reusable-extensible-python-functions-financial-data-analysis/#load_data): Load data from a CSV, Excel, or Pickle file into a pandas DataFrame.

In [373]:
from load_data import load_data

In [374]:
ticker = "XLE"
source = "Polygon"
asset_class = "Exchange_Traded_Funds"
timeframe = "day"

In [375]:
df_post_split = load_data(
    base_directory=DATA_DIR,
    ticker=f"{ticker}-postsplit",
    source=source,
    asset_class=asset_class,
    timeframe=timeframe,
    file_format="pickle",
)

df_post_split

,Date,open,high,low,close,volume,vwap,transactions,otc
0,2024-04-04 04:00:00,48.625,48.7975,48.3300,48.520,3.694129e+07,48.5901,153821,None
1,2024-04-05 04:00:00,48.720,49.2334,48.4250,49.040,3.119454e+07,48.9250,132988,None
2,2024-04-08 04:00:00,49.065,49.2050,48.6425,48.730,3.403456e+07,48.8900,125942,None
3,2024-04-09 04:00:00,48.940,49.0700,48.3900,48.745,3.533310e+07,48.6567,137980,None
4,2024-04-10 04:00:00,48.610,49.0750,48.4000,48.895,3.919965e+07,48.7887,167317,None
...,...,...,...,...,...,...,...,...,...
496,2026-03-27 04:00:00,61.530,62.7900,61.2600,62.560,5.955321e+07,62.4543,230115,None
497,2026-03-30 04:00:00,63.130,63.4600,61.7800,61.960,4.975012e+07,62.6713,252519,None
498,2026-03-31 04:00:00,62.040,62.8300,60.0350,61.260,9.476693e+07,61.3894,397629,None
499,2026-04-01 04:00:00,59.720,60.6200,58.3550,58.970,9.665290e+07,59.1639,445495,None


In [376]:
cutoff_date = df_post_split.loc[3, "Date"]
print(f"Cutoff date for split adjustment: {cutoff_date}")

Cutoff date for split adjustment: 2024-04-09 04:00:00


In [377]:
df_pre_split = load_data(
    base_directory=DATA_DIR,
    ticker=f"{ticker}-presplit",
    source=source,
    asset_class=asset_class,
    timeframe=timeframe,
    file_format="pickle",
)

print(df_pre_split.shape)
df_pre_split[df_pre_split["Date"] < cutoff_date]

(576, 9)


,Date,open,high,low,close,volume,vwap,transactions,otc
0,2023-08-02 04:00:00,86.48,86.9200,85.2000,85.93,22961517.0,85.8932,180996,None
1,2023-08-03 04:00:00,86.16,87.5800,85.6900,86.80,20816983.0,86.8191,163864,None
2,2023-08-04 04:00:00,87.42,88.2400,86.8200,86.92,21604149.0,87.5564,167009,None
3,2023-08-07 04:00:00,87.41,87.5900,86.7643,87.02,13410765.0,87.1481,114589,None
4,2023-08-08 04:00:00,85.61,87.4850,84.9650,87.45,18618558.0,86.4580,150912,None
...,...,...,...,...,...,...,...,...,...
167,2024-04-02 04:00:00,95.52,96.5400,95.1000,96.44,19335599.0,95.9267,169185,None
168,2024-04-03 04:00:00,96.80,97.2302,96.4849,97.10,14574431.0,96.9677,134716,None
169,2024-04-04 04:00:00,97.25,97.5950,96.6600,97.04,18470645.0,97.1802,153821,None
170,2024-04-05 04:00:00,97.44,98.4667,96.8500,98.08,15597269.0,97.8501,132988,None


In [378]:
SPLIT_RATIO = 2

df_pre_split_corrected = df_pre_split.copy()
df_pre_split_corrected["open"] = df_pre_split_corrected["open"] / SPLIT_RATIO
df_pre_split_corrected["high"] = df_pre_split_corrected["high"] / SPLIT_RATIO
df_pre_split_corrected["low"] = df_pre_split_corrected["low"] / SPLIT_RATIO
df_pre_split_corrected["close"] = df_pre_split_corrected["close"] / SPLIT_RATIO
df_pre_split_corrected["volume"] = df_pre_split_corrected["volume"] * SPLIT_RATIO
df_pre_split_corrected["vwap"] = df_pre_split_corrected["vwap"] / SPLIT_RATIO
df_pre_split_corrected[df_pre_split_corrected["Date"] < cutoff_date]

,Date,open,high,low,close,volume,vwap,transactions,otc
0,2023-08-02 04:00:00,43.240,43.46000,42.60000,42.965,45923034.0,42.94660,180996,None
1,2023-08-03 04:00:00,43.080,43.79000,42.84500,43.400,41633966.0,43.40955,163864,None
2,2023-08-04 04:00:00,43.710,44.12000,43.41000,43.460,43208298.0,43.77820,167009,None
3,2023-08-07 04:00:00,43.705,43.79500,43.38215,43.510,26821530.0,43.57405,114589,None
4,2023-08-08 04:00:00,42.805,43.74250,42.48250,43.725,37237116.0,43.22900,150912,None
...,...,...,...,...,...,...,...,...,...
167,2024-04-02 04:00:00,47.760,48.27000,47.55000,48.220,38671198.0,47.96335,169185,None
168,2024-04-03 04:00:00,48.400,48.61510,48.24245,48.550,29148862.0,48.48385,134716,None
169,2024-04-04 04:00:00,48.625,48.79750,48.33000,48.520,36941290.0,48.59010,153821,None
170,2024-04-05 04:00:00,48.720,49.23335,48.42500,49.040,31194538.0,48.92505,132988,None


In [379]:
display_date = df_post_split.loc[5, "Date"]

In [380]:
combined = pd.concat([df_pre_split_corrected[df_pre_split_corrected["Date"] <= cutoff_date], df_post_split], ignore_index=True)
combined[combined["Date"] <= display_date].tail(15)

,Date,open,high,low,close,volume,vwap,transactions,otc
164,2024-03-27 04:00:00,46.170,46.71000,46.09000,46.700,33960082.0,46.46015,122917,None
165,2024-03-28 04:00:00,46.975,47.29000,46.74785,47.205,34615140.0,47.08885,137994,None
166,2024-04-01 04:00:00,47.345,47.69500,46.87815,47.555,42899028.0,47.48530,162459,None
167,2024-04-02 04:00:00,47.760,48.27000,47.55000,48.220,38671198.0,47.96335,169185,None
168,2024-04-03 04:00:00,48.400,48.61510,48.24245,48.550,29148862.0,48.48385,134716,None
169,2024-04-04 04:00:00,48.625,48.79750,48.33000,48.520,36941290.0,48.59010,153821,None
170,2024-04-05 04:00:00,48.720,49.23335,48.42500,49.040,31194538.0,48.92505,132988,None
171,2024-04-08 04:00:00,49.065,49.20500,48.64250,48.730,34034558.0,48.89000,125942,None
172,2024-04-09 04:00:00,48.940,49.07000,48.39000,48.745,35333104.0,48.65665,137980,None
173,2024-04-04 04:00:00,48.625,48.79750,48.33000,48.520,36941290.0,48.59010,153821,None


In [381]:
print(f"Should drop the following duplicate rows if they exist:")
combined[combined.round(3).duplicated(subset=["Date", "open", "high", "low", "close", "volume"], keep=False)]

Should drop the following duplicate rows if they exist:


,Date,open,high,low,close,volume,vwap,transactions,otc
169,2024-04-04 04:00:00,48.625,48.79750,48.3300,48.520,36941290.0,48.59010,153821,None
170,2024-04-05 04:00:00,48.720,49.23335,48.4250,49.040,31194538.0,48.92505,132988,None
171,2024-04-08 04:00:00,49.065,49.20500,48.6425,48.730,34034558.0,48.89000,125942,None
172,2024-04-09 04:00:00,48.940,49.07000,48.3900,48.745,35333104.0,48.65665,137980,None
173,2024-04-04 04:00:00,48.625,48.79750,48.3300,48.520,36941290.0,48.59010,153821,None
174,2024-04-05 04:00:00,48.720,49.23340,48.4250,49.040,31194538.0,48.92500,132988,None
175,2024-04-08 04:00:00,49.065,49.20500,48.6425,48.730,34034558.0,48.89000,125942,None
176,2024-04-09 04:00:00,48.940,49.07000,48.3900,48.745,35333104.0,48.65670,137980,None


In [382]:
combined = pd.concat([df_pre_split_corrected[df_pre_split_corrected["Date"] <= cutoff_date], df_post_split], ignore_index=True).round(3).drop_duplicates(subset=["Date", "open", "high", "low", "close", "volume"], keep="last")
combined[combined["Date"] <= display_date].tail(15)

,Date,open,high,low,close,volume,vwap,transactions,otc
160,2024-03-21 04:00:00,46.105,46.395,45.960,46.300,23334194.0,46.261,102043,None
161,2024-03-22 04:00:00,46.320,46.435,46.087,46.200,17948808.0,46.192,90997,None
162,2024-03-25 04:00:00,46.360,46.940,46.350,46.630,35074380.0,46.667,132760,None
163,2024-03-26 04:00:00,46.675,46.780,46.180,46.270,35373850.0,46.374,130682,None
164,2024-03-27 04:00:00,46.170,46.710,46.090,46.700,33960082.0,46.460,122917,None
165,2024-03-28 04:00:00,46.975,47.290,46.748,47.205,34615140.0,47.089,137994,None
166,2024-04-01 04:00:00,47.345,47.695,46.878,47.555,42899028.0,47.485,162459,None
167,2024-04-02 04:00:00,47.760,48.270,47.550,48.220,38671198.0,47.963,169185,None
168,2024-04-03 04:00:00,48.400,48.615,48.242,48.550,29148862.0,48.484,134716,None
173,2024-04-04 04:00:00,48.625,48.798,48.330,48.520,36941290.0,48.590,153821,None


In [383]:
combined.reset_index(drop=True, inplace=True)
combined

,Date,open,high,low,close,volume,vwap,transactions,otc
0,2023-08-02 04:00:00,43.240,43.460,42.600,42.965,4.592303e+07,42.947,180996,None
1,2023-08-03 04:00:00,43.080,43.790,42.845,43.400,4.163397e+07,43.410,163864,None
2,2023-08-04 04:00:00,43.710,44.120,43.410,43.460,4.320830e+07,43.778,167009,None
3,2023-08-07 04:00:00,43.705,43.795,43.382,43.510,2.682153e+07,43.574,114589,None
4,2023-08-08 04:00:00,42.805,43.742,42.482,43.725,3.723712e+07,43.229,150912,None
...,...,...,...,...,...,...,...,...,...
665,2026-03-27 04:00:00,61.530,62.790,61.260,62.560,5.955321e+07,62.454,230115,None
666,2026-03-30 04:00:00,63.130,63.460,61.780,61.960,4.975012e+07,62.671,252519,None
667,2026-03-31 04:00:00,62.040,62.830,60.035,61.260,9.476693e+07,61.389,397629,None
668,2026-04-01 04:00:00,59.720,60.620,58.355,58.970,9.665290e+07,59.164,445495,None


In [384]:
# Export the combined DataFrame to a pickle file
combined.to_pickle(DATA_DIR / source / asset_class / timeframe / f"{ticker}.pkl")